# Day 19 — Cross-validation and robust validation practices

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Cross-validation fundamentals

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-019/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 23 test students are set aside first, as on Days 17 and 18; everything today happens inside the other 127
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)
model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=1000))])

# One split, twenty times: the same model, the same 127 students, a different random 80/20 split each time
accuracies = []
for seed in range(20):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    accuracies.append(model.fit(X_tr, y_tr).score(X_val, y_val))
print("Validation students per split:", len(X_val))
print("Accuracy on 20 different splits:")
print(np.round(accuracies[:10], 2))
print(np.round(accuracies[10:], 2))
print(f"Lowest {min(accuracies):.3f} | highest {max(accuracies):.3f} | mean {np.mean(accuracies):.3f}")

Validation students per split: 26
Accuracy on 20 different splits:
[0.85 0.92 0.73 0.92 0.73 0.73 0.69 0.58 0.73 0.81]
[0.85 0.65 0.81 0.69 0.73 0.77 0.77 0.65 0.81 0.77]
Lowest 0.577 | highest 0.923 | mean 0.760


In [ ]:
# The lesson's five fold scores: mean and spread
fold_scores = np.array([0.88, 0.91, 0.86, 0.90, 0.85])
print("Mean:", fold_scores.mean().round(3))
print("Standard deviation (np.std, as scikit-learn reports):", fold_scores.std().round(4))
print("Sample standard deviation (ddof=1):                 ", fold_scores.std(ddof=1).round(4))

Mean: 0.88
Standard deviation (np.std, as scikit-learn reports): 0.0228
Sample standard deviation (ddof=1):                  0.0255


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# 5-fold cross-validation: every student is validated once and trained on four times
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")
print("Fold scores:", scores.round(3))
print(f"Mean accuracy {scores.mean():.3f} | standard deviation {scores.std():.3f}")
print("Validation fold sizes:", [len(v) for _, v in kfold.split(X)])

Fold scores: [0.846 0.692 0.76  0.64  0.92 ]
Mean accuracy 0.772 | standard deviation 0.101
Validation fold sizes: [26, 26, 25, 25, 25]


In [ ]:
from sklearn.model_selection import StratifiedKFold

# 37 of the 127 students fail (29%). How many failing students land in each validation fold?
print(f"Failing overall: {(y == 0).sum()} of {len(y)} ({(y == 0).mean():.0%})")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    fails = [int((y.iloc[v] == 0).sum()) for _, v in splitter.split(X, y)]
    print(f"{name:<10} failing per fold {fails}")

Failing overall: 37 of 127 (29%)
KFold      failing per fold [5, 7, 10, 8, 7]
Stratified failing per fold [8, 8, 7, 7, 7]


In [ ]:
from sklearn.metrics import make_scorer, f1_score

# F1 for fail (the class we want to catch) per fold, with plain and stratified 5-fold
fail_f1 = make_scorer(f1_score, pos_label=0)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    s = cross_val_score(model, X, y, cv=splitter, scoring=fail_f1)
    print(f"{name:<10} {s.round(2)} mean {s.mean():.3f} std {s.std():.3f}")

KFold      [0.6  0.5  0.62 0.31 0.86] mean 0.578 std 0.179
Stratified [0.59 0.67 0.36 0.73 0.75] mean 0.619 std 0.140


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Preprocessing leakage: a scaler fitted on all 127 students before cross-validation has seen every validation fold
knn = KNeighborsClassifier()
leaky = cross_val_score(knn, StandardScaler().fit_transform(X), y, cv=skf).mean()
inside = cross_val_score(Pipeline([("scaler", StandardScaler()), ("classifier", knn)]), X, y, cv=skf).mean()
print(f"Scaler fitted before CV:  {leaky:.3f}")
print(f"Scaler inside the pipeline: {inside:.3f}")

Scaler fitted before CV:  0.771
Scaler inside the pipeline: 0.795


## 2. Stratified and repeated K-fold

In [ ]:
# Stratified 5-fold on our 127 students: what each validation fold holds
print(f"All 127: {(y == 1).sum()} pass, {(y == 0).sum()} fail ({(y == 0).mean():.0%} fail)")
for k, (_, val) in enumerate(skf.split(X, y), 1):
    print(f"Fold {k}: {len(val)} students, {(y.iloc[val] == 1).sum()} pass, {(y.iloc[val] == 0).sum()} fail "
          f"({(y.iloc[val] == 0).mean():.0%})")

All 127: 90 pass, 37 fail (29% fail)
Fold 1: 26 students, 18 pass, 8 fail (31%)
Fold 2: 26 students, 18 pass, 8 fail (31%)
Fold 3: 25 students, 18 pass, 7 fail (28%)
Fold 4: 25 students, 18 pass, 7 fail (28%)
Fold 5: 25 students, 18 pass, 7 fail (28%)


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

# One stratified 5-fold run depends on how the students were shuffled: ten shuffles, ten different estimates
one_run = [cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed), scoring=fail_f1).mean()
           for seed in range(10)]
print("Mean F1 for fail, ten single 5-fold runs:")
print(np.round(one_run[:5], 3))
print(np.round(one_run[5:], 3))
print(f"Range {min(one_run):.3f} to {max(one_run):.3f}")

# Repeated stratified 5-fold: 5 folds x 3 repetitions = 15 fits, averaged
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores = cross_val_score(model, X, y, cv=rskf, scoring=fail_f1)
print(f"RepeatedStratifiedKFold: {len(scores)} evaluations, mean {scores.mean():.3f}, std {scores.std():.3f}")

Mean F1 for fail, ten single 5-fold runs:
[0.605 0.542 0.556 0.615 0.607]
[0.582 0.587 0.568 0.538 0.559]
Range 0.538 to 0.615
RepeatedStratifiedKFold: 15 evaluations, mean 0.591, std 0.199


In [ ]:
# 3, 5 and 10 folds, each repeated 10 times (so the comparison isn't one lucky shuffle). Cost = number of fits
print("folds  train  failing/fold  mean F1  std    fits")
for k in [3, 5, 10]:
    cv_k = RepeatedStratifiedKFold(n_splits=k, n_repeats=10, random_state=42)
    s = cross_val_score(model, X, y, cv=cv_k, scoring=fail_f1)
    fails = sorted({int((y.iloc[v] == 0).sum()) for _, v in cv_k.split(X, y)})
    print(f"{k:>5}  {1 - 1 / k:>5.0%}  {fails[0]:>6}-{fails[-1]:<5}  {s.mean():>7.3f}  {s.std():.3f}  {len(s):>4}")

folds  train  failing/fold  mean F1  std    fits


    3    67%      12-13       0.582  0.138    30


    5    80%       7-8        0.592  0.162    50


   10    90%       3-4        0.575  0.217   100


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import precision_score, recall_score

# Several metrics in one cross-validation run. "fail" is our positive class, so precision, recall and F1 say pos_label=0;
# ROC-AUC and accuracy don't depend on which class is positive
scoring = {"accuracy": "accuracy", "roc_auc": "roc_auc",
           "precision_fail": make_scorer(precision_score, pos_label=0),
           "recall_fail": make_scorer(recall_score, pos_label=0), "f1_fail": fail_f1}
results = cross_validate(model, X, y, cv=skf, scoring=scoring, return_train_score=True)
print(f"{'metric':<15}{'validation':>11}{'training':>10}")
for name in scoring:
    print(f"{name:<15}{results['test_' + name].mean():>11.3f}{results['train_' + name].mean():>10.3f}")

metric          validation  training
accuracy             0.796     0.789
roc_auc              0.835     0.866
precision_fail       0.687     0.666
recall_fail          0.593     0.554
f1_fail              0.619     0.604


## 3. Validation for time-series data

In [ ]:
# A synthetic monthly series (seeded): an upward trend of 15 a month plus noise. Five years, 60 months
dates = pd.date_range(start="2025-01-01", periods=60, freq="MS")
rng = np.random.default_rng(42)
monthly = pd.DataFrame({"date": dates,
                        "transactions": 1000 + np.arange(60) * 15 + rng.normal(0, 80, 60)}).set_index("date")
print(monthly.head().round(1))

            transactions
date                    
2025-01-01        1024.4
2025-02-01         931.8
2025-03-01        1090.0
2025-04-01        1120.2
2025-05-01         903.9


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# TimeSeriesSplit: every validation block comes after its training block, and training grows (an expanding window)
def show(splitter, data):
    for fold, (tr, va) in enumerate(splitter.split(data), 1):
        t, v = data.index[tr], data.index[va]
        print(f"Fold {fold}: train {t.min():%Y-%m}..{t.max():%Y-%m} ({len(tr):>2}) | "
              f"valid {v.min():%Y-%m}..{v.max():%Y-%m} ({len(va)})")

show(TimeSeriesSplit(n_splits=5), monthly)

Fold 1: train 2025-01..2025-10 (10) | valid 2025-11..2026-08 (10)
Fold 2: train 2025-01..2026-08 (20) | valid 2026-09..2027-06 (10)
Fold 3: train 2025-01..2027-06 (30) | valid 2027-07..2028-04 (10)
Fold 4: train 2025-01..2028-04 (40) | valid 2028-05..2029-02 (10)
Fold 5: train 2025-01..2029-02 (50) | valid 2029-03..2029-12 (10)


In [ ]:
# A rolling window: max_train_size keeps only the most recent 24 months for training
print("Rolling window (24 months):")
show(TimeSeriesSplit(n_splits=5, max_train_size=24), monthly)
# gap=2: skip the 2 months between training and validation (for example, labels that arrive two months late)
print("Expanding window with gap=2:")
show(TimeSeriesSplit(n_splits=5, gap=2), monthly)

Rolling window (24 months):
Fold 1: train 2025-01..2025-10 (10) | valid 2025-11..2026-08 (10)
Fold 2: train 2025-01..2026-08 (20) | valid 2026-09..2027-06 (10)
Fold 3: train 2025-07..2027-06 (24) | valid 2027-07..2028-04 (10)
Fold 4: train 2026-05..2028-04 (24) | valid 2028-05..2029-02 (10)
Fold 5: train 2027-03..2029-02 (24) | valid 2029-03..2029-12 (10)
Expanding window with gap=2:
Fold 1: train 2025-01..2025-08 ( 8) | valid 2025-11..2026-08 (10)
Fold 2: train 2025-01..2026-06 (18) | valid 2026-09..2027-06 (10)
Fold 3: train 2025-01..2027-04 (28) | valid 2027-07..2028-04 (10)
Fold 4: train 2025-01..2028-02 (38) | valid 2028-05..2029-02 (10)
Fold 5: train 2025-01..2028-12 (48) | valid 2029-03..2029-12 (10)


In [ ]:
from sklearn.linear_model import LinearRegression

# Lag features: last month's, two months ago and three months ago. The first 3 months have no history, so they go
lagged = monthly.copy()
for k in (1, 2, 3):
    lagged[f"lag_{k}"] = lagged["transactions"].shift(k)
lagged = lagged.dropna()
X_ts, y_ts = lagged[["lag_1", "lag_2", "lag_3"]], lagged["transactions"]
forecaster = Pipeline([("scaler", StandardScaler()), ("regressor", LinearRegression())])

tscv = TimeSeriesSplit(n_splits=5)
mae = -cross_val_score(forecaster, X_ts, y_ts, cv=tscv, scoring="neg_mean_absolute_error")
print("Rows after the lags:", len(lagged))
print("Chronological MAE per fold:", mae.round(1), "| mean", mae.mean().round(1))

# The same models scored on shuffled folds, which let them use later months to predict earlier ones
from sklearn.ensemble import RandomForestRegressor
print(f"{'mean MAE':<18}{'chronological':>14}{'shuffled':>10}")
for name, est in [("Linear regression", forecaster), ("Random forest", RandomForestRegressor(n_estimators=200, random_state=42))]:
    chrono = -cross_val_score(est, X_ts, y_ts, cv=tscv, scoring="neg_mean_absolute_error").mean()
    shuffled = -cross_val_score(est, X_ts, y_ts, cv=KFold(5, shuffle=True, random_state=42),
                                scoring="neg_mean_absolute_error").mean()
    print(f"{name:<18}{chrono:>14.1f}{shuffled:>10.1f}")

Rows after the lags: 57
Chronological MAE per fold: [85.7 61.2 85.2 36.5 69.2] | mean 67.5
mean MAE           chronological  shuffled


Linear regression           67.5      70.9


Random forest               96.9      81.8


In [ ]:
# Temporal leakage in a feature: a 3-month rolling average that includes the month being predicted
leaky_ts = lagged.assign(avg_3=monthly["transactions"].rolling(3).mean())           # months t-2, t-1 and t itself
safe_ts = lagged.assign(avg_3=monthly["transactions"].shift(1).rolling(3).mean())   # months t-3, t-2 and t-1
cols = ["lag_1", "lag_2", "lag_3", "avg_3"]
for name, data in [("avg_3 includes this month (leaks)", leaky_ts), ("avg_3 shifted to past months", safe_ts)]:
    m = -cross_val_score(forecaster, data[cols], data["transactions"], cv=tscv, scoring="neg_mean_absolute_error")
    print(f"{name:<34} mean MAE {m.mean():.1f}")

avg_3 includes this month (leaks)  mean MAE 0.0
avg_3 shifted to past months       mean MAE 67.5


## 4. Group and nested cross-validation

In [ ]:
import warnings
from sklearn.model_selection import GroupKFold

# Twelve transactions from six customers, two each. GroupKFold keeps each customer in one fold
X_tx = np.array([[5000], [8000], [50000], [45000], [2000], [3000], [7000], [9000], [60000], [55000], [4000], [3500]])
y_tx = np.array([0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0])
groups = np.array(["C101", "C101", "C102", "C102", "C103", "C103", "C104", "C104", "C105", "C105", "C106", "C106"])
gkf = GroupKFold(n_splits=3)
for k, (tr, va) in enumerate(gkf.split(X_tx, y_tx, groups=groups), 1):
    print(f"Fold {k}: validate {', '.join(sorted(set(groups[va])))} | fraud rows: validation {int(y_tx[va].sum())},"
          f" training {int(y_tx[tr].sum())}")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                         # the failed fit is the point: see the nan below
    scores = cross_val_score(LogisticRegression(), X_tx, y_tx, cv=gkf, groups=groups, scoring="accuracy")
print("Fold scores:", scores)

Fold 1: validate C103, C106 | fraud rows: validation 0, training 4
Fold 2: validate C102, C105 | fraud rows: validation 4, training 0
Fold 3: validate C101, C104 | fraud rows: validation 0, training 4
Fold scores: [ 1. nan  1.]


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# StratifiedGroupKFold: keep customers together *and* spread the fraud customers across folds
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
for k, (tr, va) in enumerate(sgkf.split(X_tx, y_tx, groups=groups), 1):
    assert set(groups[tr]).isdisjoint(groups[va])          # no customer on both sides
    print(f"Fold {k}: validate {', '.join(sorted(set(groups[va])))} | fraud in validation {y_tx[va].tolist()}")

Fold 1: validate C101, C105 | fraud in validation [0, 0, 1, 1]
Fold 2: validate C102, C103 | fraud in validation [1, 1, 0, 0]
Fold 3: validate C104, C106 | fraud in validation [0, 0, 0, 0]


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# The leak, at scale: 60 synthetic customers, 10 transactions each (seeded). Each customer has their own typical
# amount and spending hour, and fraud runs in some customers' accounts. Can a model spot fraud for a *new* customer?
rng = np.random.default_rng(7)
rows = []
for c in range(60):
    base, hour, risky = rng.uniform(1000, 60000), rng.uniform(8, 22), rng.random() < 0.3
    for _ in range(10):
        rows.append({"customer": f"C{c:03d}", "amount": base * rng.lognormal(0, 0.15),
                     "hour": hour + rng.normal(0, 1), "fraud": int(rng.random() < (0.6 if risky else 0.05))})
tx = pd.DataFrame(rows)
print(tx.shape, "| customers", tx["customer"].nunique(), f"| fraud {tx['fraud'].mean():.1%}")
forest = RandomForestClassifier(n_estimators=200, random_state=42)
Xg, yg, gg = tx[["amount", "hour"]], tx["fraud"], tx["customer"]
for name, splitter, grp in [("StratifiedKFold (records)", StratifiedKFold(5, shuffle=True, random_state=42), None),
                            ("GroupKFold (customers)", GroupKFold(n_splits=5), gg)]:
    s = cross_val_score(forest, Xg, yg, cv=splitter, groups=grp, scoring="roc_auc")
    print(f"{name:<26} ROC-AUC {s.mean():.3f}")

(600, 4) | customers 60 | fraud 20.3%


StratifiedKFold (records)  ROC-AUC 0.614


GroupKFold (customers)     ROC-AUC 0.465


In [ ]:
from sklearn.model_selection import GridSearchCV

# Nested CV on our 127 students. Inner 3-fold picks k (neighbours); outer 5-fold scores the picked model on folds it never saw.
pipe = Pipeline([("scaler", StandardScaler()), ("classifier", KNeighborsClassifier())])
param_grid = {"classifier__n_neighbors": [1, 3, 5, 7, 9, 15, 25]}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
search = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring=fail_f1)

# Not nested: tune and report the same cross-validation score (optimistic: the best of 7 tries on the same folds)
search.fit(X, y)
print(f"Not nested: best k {search.best_params_['classifier__n_neighbors']}, CV F1 for fail {search.best_score_:.3f}")
# Nested: the whole search runs inside each outer training fold
nested = cross_validate(search, X, y, cv=outer_cv, scoring=fail_f1, return_estimator=True)
print("Nested outer scores:", nested["test_score"].round(3), f"| mean {nested['test_score'].mean():.3f}")
print("k chosen in each outer fold:", [e.best_params_["classifier__n_neighbors"] for e in nested["estimator"]])
print("Fits: 7 settings x 3 inner folds x 5 outer folds = 105, plus 5 refits")

Not nested: best k 3, CV F1 for fail 0.695


Nested outer scores: [0.667 0.533 0.588 0.333 0.5  ] | mean 0.524
k chosen in each outer fold: [1, 1, 5, 3, 5]
Fits: 7 settings x 3 inner folds x 5 outer folds = 105, plus 5 refits


## 5. Model evaluation with cross-validation

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Accuracy alone: a "model" that always predicts the majority class
fraud_true = np.array([1] * 500 + [0] * 9500)                  # 500 fraud among 10,000
print(f"Always 'legitimate': accuracy {accuracy_score(fraud_true, np.zeros(10000)):.2f}, "
      f"fraud caught {recall_score(fraud_true, np.zeros(10000)):.0%}")
# The same on our students: always predict pass
always_pass = cross_validate(DummyClassifier(strategy="most_frequent"), X, y, cv=skf,
                             scoring={"accuracy": "accuracy", "recall_fail": make_scorer(recall_score, pos_label=0)})
print(f"Always 'pass' (students): accuracy {always_pass['test_accuracy'].mean():.3f}, "
      f"fails caught {always_pass['test_recall_fail'].mean():.0%}")

Always 'legitimate': accuracy 0.95, fraud caught 0%
Always 'pass' (students): accuracy 0.709, fails caught 0%


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

# The fraud confusion matrix: TP 350, FN 150, FP 200, TN 9,300
tp, fn, fp, tn = 350, 150, 200, 9300
precision, recall = tp / (tp + fp), tp / (tp + fn)
print(f"Accuracy {(tp + tn) / (tp + tn + fp + fn):.3f} | precision {precision:.3f} | recall {recall:.3f} | "
      f"F1 {2 * precision * recall / (precision + recall):.3f}")

# Eight predictions, as scikit-learn computes them (1 = fraud)
y_true = [1, 1, 1, 0, 0, 0, 1, 0]
y_pred = [1, 1, 0, 0, 1, 0, 1, 0]
y_scores = [0.95, 0.80, 0.40, 0.10, 0.70, 0.20, 0.85, 0.15]
print("Accuracy", accuracy_score(y_true, y_pred), "| precision", precision_score(y_true, y_pred))
print("Recall", recall_score(y_true, y_pred), "| F1", f1_score(y_true, y_pred))
print("Confusion matrix [[TN FP] [FN TP]]:", confusion_matrix(y_true, y_pred).tolist())
print("ROC-AUC from scores:", roc_auc_score(y_true, y_scores))
print("ROC-AUC from hard 0/1 predictions (a mistake):", roc_auc_score(y_true, y_pred))

Accuracy 0.965 | precision 0.636 | recall 0.700 | F1 0.667
Accuracy 0.75 | precision 0.75
Recall 0.75 | F1 0.75
Confusion matrix [[TN FP] [FN TP]]: [[3, 1], [1, 3]]
ROC-AUC from scores: 0.9375
ROC-AUC from hard 0/1 predictions (a mistake): 0.75


In [ ]:
from sklearn.metrics import average_precision_score

# Five metrics per fold on our students, "fail" as the positive class. PR-AUC (average precision) needs P(fail)
fail_scoring = {"accuracy": "accuracy",
                "precision": make_scorer(precision_score, pos_label=0),
                "recall": make_scorer(recall_score, pos_label=0), "f1": fail_f1,
                "roc_auc": "roc_auc",
                "pr_auc": make_scorer(average_precision_score, response_method="predict_proba", pos_label=0)}
res = cross_validate(model, X, y, cv=skf, scoring=fail_scoring, return_train_score=True)
table = pd.DataFrame({m: res[f"test_{m}"] for m in fail_scoring}, index=[f"Fold {k}" for k in range(1, 6)])
table.loc["Mean"], table.loc["Std"] = table.iloc[:5].mean(), table.iloc[:5].std(ddof=0)
print(table.round(3).to_string())
print(f"PR-AUC at random: {(y == 0).mean():.3f} (the share of failing students)")

        accuracy  precision  recall     f1  roc_auc  pr_auc
Fold 1     0.731      0.556   0.625  0.588    0.757   0.568
Fold 2     0.808      0.714   0.625  0.667    0.847   0.762
Fold 3     0.720      0.500   0.286  0.364    0.825   0.619
Fold 4     0.880      1.000   0.571  0.727    0.794   0.765
Fold 5     0.840      0.667   0.857  0.750    0.952   0.898
Mean       0.796      0.687   0.593  0.619    0.835   0.722
Std        0.062      0.174   0.183  0.140    0.066   0.117
PR-AUC at random: 0.291 (the share of failing students)


In [ ]:
# Two models on identical folds, with training scores to spot overfitting
models = {"Logistic regression": model,
          "Random forest": RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)}
rows = {}
for name, est in models.items():
    r = cross_validate(est, X, y, cv=skf, scoring={"f1": fail_f1, "roc_auc": "roc_auc"}, return_train_score=True)
    rows[name] = {"F1 val": f"{r['test_f1'].mean():.3f}±{r['test_f1'].std():.3f}",
                  "AUC val": f"{r['test_roc_auc'].mean():.3f}±{r['test_roc_auc'].std():.3f}",
                  "F1 train": f"{r['train_f1'].mean():.3f}",
                  "gap": f"{r['train_f1'].mean() - r['test_f1'].mean():.3f}"}
print(pd.DataFrame(rows).T.to_string())

                          F1 val      AUC val F1 train     gap
Logistic regression  0.619±0.140  0.835±0.066    0.604  -0.015
Random forest        0.633±0.074  0.812±0.052    0.990   0.357
